# 08 — A4 를 제대로 재기: 전체 val 로짓 (학습 없음, ~30분)

## 묻는 것 하나

STEP 16 에서 **A4(농포) recall 0.264** 로 최악인데 원인이 미측정입니다.
로컬(VL01)에서 여기까지 왔습니다:

* **로짓 보정(prior)으로는 못 고칩니다** — A4 recall 은 오르는데 macro-F1 이
  +0.0021 뿐이고 A2·A3 가 정확히 그만큼 내려갑니다 (STEP 18, 개입 실험)
* 즉 **결정 규칙이 아니라 표현**의 문제입니다

⚠️ **그런데 그 분석은 전부 VL01 로만 했습니다** — A4 의 **88%는 TL02** 에 있고,
VL01 의 A4 는 나머지보다 **1.7배 큽니다** (196px vs 113px). 그래서 지금 숫자를
STEP 16 의 0.264 와 나란히 놓으면 안 됩니다.

**이 노트북은 학습을 하지 않습니다.** STEP 16 가중치로 **전체 val** 을 한 번
추론해 로짓만 뽑습니다. 그 뒤 분석은 전부 로컬에서 공짜로 돕니다.

## 붙일 것 (Add Input)

| 입력 | 필수 | 비고 |
|---|---|---|
| `m2.5` 크롭 (365,428행) | ✅ | 2단계 입력 |
| **STEP 16 release** | ✅ | `stage2_.../best.pt` + `stage1_threshold.json` |
| `f320` 크롭 | ❌ | 1단계도 볼 때만. 없으면 `--stages 2` |

⚠️ release 를 **zip 으로** 올리셨으면 캐글이 안 풀어줍니다 — 아래 2번 셀이 풉니다.

## 돌리는 법

우측 상단 **[Save Version] → Save & Run All (Commit)**.
브라우저를 닫아도 끝까지 돌고 출력이 보존됩니다.

In [ ]:
# ── 브랜치 지정 — **0번 셀보다 먼저** 돌려야 합니다 ─────────────────
# ⚠️ 0번 셀은 브랜치를 못 찾으면 'main' 을 clone 합니다. 이 노트북이 쓰는
#    tools/local_logits.py 는 작업 브랜치에만 있으므로, main 을 받으면
#    '그런 파일이 없습니다' 로 죽습니다.
import os
os.environ["DOG_SKIN_BRANCH"] = "claude/dog-disease-diagnosis-model-1s6jtf"   # ← main 에 머지했으면 "main" 으로
print("clone 할 브랜치:", os.environ["DOG_SKIN_BRANCH"])


In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-28.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 1. release 찾아서 풀기

캐글 입력에 `checkpoints/` 폴더가 보여야 `import_previous_run()` 이 가져갑니다.
zip 으로 올렸으면 여기서 풉니다 (입력은 읽기 전용이라 `/kaggle/working` 으로).

In [ ]:
import os, zipfile, shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
work = Path("/kaggle/working/release_unzipped")

def has_ckpt(p: Path) -> bool:
    return any(p.rglob("best.pt"))

# 1) 이미 폴더로 풀려 있나
ready = [d for d in INPUT.glob("*") if d.is_dir() and has_ckpt(d)]
if ready:
    print("✅ 이미 폴더로 붙어 있습니다:")
    for d in ready:
        print("  ", d)
else:
    # 2) zip 을 찾아 풉니다
    zips = [z for z in INPUT.rglob("*.zip")]
    print(f"zip 후보 {len(zips)}개:", [z.name for z in zips])
    if not zips:
        raise SystemExit(
            "❌ 가중치도 zip 도 없습니다.\n"
            "   release 를 Private 데이터셋으로 만들어 Add Input 하세요.\n"
            f"   지금 붙어 있는 것: {[d.name for d in INPUT.glob('*')]}")
    work.mkdir(parents=True, exist_ok=True)
    for z in zips:
        with zipfile.ZipFile(z) as f:
            names = f.namelist()
            if not any(n.endswith("best.pt") for n in names):
                print(f"  ⏭️  {z.name} — best.pt 가 없어 건너뜁니다")
                continue
            print(f"  📦 {z.name} 풀는 중 ({len(names):,} 항목)")
            f.extractall(work)
    if not has_ckpt(work):
        raise SystemExit("❌ zip 을 풀었는데 best.pt 가 없습니다. 올린 내용을 확인하세요.")
    print("✅ 풀었습니다:", work)
    for p in sorted(work.rglob("best.pt")):
        print(f"   {p.relative_to(work)}  {p.stat().st_size/1e6:.0f} MB")


## 2. 전체 val 로짓 뽑기

⚠️ 스크립트가 **행 수가 300,000 미만이면 경고**합니다 — 옛 45,885행 크롭이
붙었다는 뜻이니 그때 멈추고 입력을 바꾸세요.

⚠️ `f320` 을 안 붙였으면 `--stages 2` 그대로 두세요. A4 는 2단계 얘기입니다.

In [ ]:
import subprocess, sys, os
OUT = "/kaggle/working/step18_full"
# release 를 푼 곳이 있으면 그 경로를 환경변수로 알려줍니다 (자동 탐색이 실패할 때 대비)
env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
cmd = [sys.executable, "tools/local_logits.py",
       "--chunk", "all", "--stages", "2", "--out", OUT]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd=DIR, env=env, text=True)
print("종료코드:", r.returncode)
if r.returncode != 0:
    raise SystemExit("❌ 실패 — 위 로그를 그대로 복사해서 공유해주세요.")


## 3. 결과 확인 + 내려받기

`stage2_logits.npz` + `stage2_rows.parquet` 만 있으면 이후 분석은 전부
로컬에서 됩니다. **수 MB 밖에 안 됩니다.**

In [ ]:
import shutil, os
from pathlib import Path
OUT = Path("/kaggle/working/step18_full")
tot = 0
for f in sorted(OUT.iterdir()):
    mb = f.stat().st_size / 1e6
    tot += mb
    print(f"  {f.name:32} {mb:8.2f} MB")
print(f"  {'합계':32} {tot:8.2f} MB")

shutil.make_archive("/kaggle/working/step18_full", "zip", OUT)
print("\n✅ /kaggle/working/step18_full.zip — 이 파일을 내려받아 공유하세요.")

# 바로 눈으로 확인 — 전체 val 기준 클래스별 recall
import numpy as np, sys
sys.path.insert(0, DIR)
z = np.load(OUT / "stage2_logits.npz", allow_pickle=False)
y, pred = z["y"], z["logits"].argmax(1)
classes = [str(c) for c in z["classes"]]
print(f"\n전체 val 병변 {len(y):,}장")
from sklearn.metrics import f1_score
print(f"macro-F1 {f1_score(y, pred, average='macro', zero_division=0):.4f}"
      "   (STEP 16 val 0.5999 와 같은 조건이어야 합니다)")
for i, c in enumerate(classes):
    m = y == i
    print(f"  {c}  n={int(m.sum()):>6,}  recall {float((pred[m]==i).mean()):.3f}")


## 다음

`step18_full.zip` 을 공유하면 로컬에서 이어서 돕니다 (GPU 0):

| 무엇 | 도구 |
|---|---|
| 흩어짐 / 짝 혼동 + 쏠림 lift | `errors.class_dispersion()` |
| 로짓 보정 재검증 (VL01 에선 기각) | `tools/logit_adjust.py` |
| A4 recall vs 병변 크기 | `tools/lesion_size_stats.py` |
| 클래스별 recall CI | `evaluate.bootstrap_ci(metric="recall")` |

⚠️ **holdout 은 열지 않습니다.** 판정은 val 로 합니다 (작업 규칙).